In [2]:
import os
print(os.getcwd())
os.listdir()

C:\Users\adhamz\OneDrive\Desktop\RegAnProject


['.git',
 '.ipynb_checkpoints',
 'Commit.csv',
 'EventSubset.ipynb',
 'Forcedeck.csv',
 'Nemo.csv',
 'RegAn1.6.1.ipynb',
 'RegAn1.6.ipynb',
 'RegAn1.7.ipynb']

In [3]:
import pandas as pd

commit = pd.read_csv("Commit.csv")

races = pd.read_csv("Nemo.csv")

forcedecks = pd.read_csv("Forcedeck.csv")

In [4]:
#Get rid of rows that will mess with signal (non-real races)
races = races[  
~races["CompetitionName"].isin(["Training", "Certification 2026", "CERTIFICATION 2025", "CERTIFICATION 2023", "CERTIFICATION 2022"])  
]

In [6]:
commit["athlete_id"] = commit["About"]
commit["timestamp"] = pd.to_datetime(commit["workout_date_time"], errors = "coerce")

races["athlete_id"] = races["AthleteName"]
races["race_date"] = pd.to_datetime(races["Date"], errors = "coerce", dayfirst=True)

commit = commit.copy()
races = races.copy()

In [7]:
fd_cmj = forcedecks[forcedecks["Test Type"] == "CMJ"].copy()

fd_cmj["athlete_id"] = fd_cmj["About"]
fd_cmj["timestamp"] = pd.to_datetime(fd_cmj["Date"], dayfirst=True, utc=True, errors="coerce")

In [8]:
min_commit = commit["timestamp"].min()
races_in_range = races[pd.to_datetime(races["race_date"], utc = True) >= min_commit].copy()

In [9]:
import numpy as np

commit["session_day"] = commit["timestamp"].dt.floor("D")
commit["session_block"] = np.where(commit["timestamp"].dt.hour < 12, "AM", "PM")

commit = commit.drop_duplicates(subset=["athlete_id", "session_day", "session_block", "workout_total_distance"])

In [10]:
Intensity_Cols = ["intensity_anaerobic_capacity", "intensity_aerobic_power", "intensity_anaerobic_power", "intensity_speed"]

Stroke_map = {
    "Freestyle": "stroke_free",
    "Backstroke": "stroke_back",
    "Breaststroke": "stroke_breast",
    "Butterfly": "stroke_fly",
    "Medley": "stroke_im",}

In [11]:
races_in_range["race_stroke"] = races_in_range["RaceEventSwimmingStyleName"]

In [12]:
def training_distance_features(commit, athlete_id, race_date, race_stroke):

    race_date = pd.to_datetime(race_date, utc = True)
    df = commit[commit["athlete_id"] == athlete_id]

    features = {}

    stroke_col = Stroke_map.get(race_stroke)

    for days in [10, 40]:
        start = race_date - pd.Timedelta(days=days)
        w = df[(df["timestamp"] >= start ) & (df["timestamp"] < race_date)]

        features[f"distance_{days}d"] = w["workout_total_distance"].sum()
        features[f"sessions_{days}d"] = w["workout_total_distance"].count()

        for col in Intensity_Cols:
            if col in w.columns:
                features[f"{col}_{days}d"] = w[col].sum()
            else:
                features[f"{col}_{days}d"] = 0

        #Stroke specific metres (dependant on race stroke)

        if stroke_col and stroke_col in w.columns:
            features[f"distance_{race_stroke}_{days}d"] = w[stroke_col].sum()
        else:
            features[f"distance_{race_stroke}_{days}d"] = 0

        #Taper ratio:
        d10 = features.get("distance_10d", 0)
        d40 = features.get("distance_40d", 0)

        features["taper_ratio"] = round((d10 / d40) * 100, 2) if d40 not in [0, None] else 0

    return features

In [16]:
def race_row(r):
    features = training_distance_features(commit, r["athlete_id"], r["race_date"], r["race_stroke"])
    
    features["race_phase"] = r["PhaseName"]
    features["race_stroke"] = r["RaceEventSwimmingStyleName"]
    features["race_distance"] = r["RaceEventDistance"]
    features["pool_length"] = r["PoolLength"]
    features["race_time"] = r["RaceTime"]
    features["race"] = r["RaceEventName"]
    
    cmj_df = fd_cmj[fd_cmj["athlete_id"] == r["athlete_id"]]
    r["race_date"] = pd.to_datetime(r["race_date"], utc=True)
    
    for days in [10, 40]:
        start = r["race_date"] - pd.Timedelta(days=days)
    
        w_cmj = cmj_df[
            (cmj_df["timestamp"] >= start) &
            (cmj_df["timestamp"] < r["race_date"])
        ]
        
        features[f"cmj_tests_{days}d"] = len(w_cmj)
        features[f"cmj_jump_height_mean_{days}d"] = w_cmj["Jump Height (Flight Time) [cm]"].mean()
        features[f"cmj_jump_height_max_{days}d"] = w_cmj["Jump Height (Flight Time) [cm]"].max()
    
        features[f"cmj_peak_power_mean_{days}d"] = w_cmj["Peak Power / BM [W/kg]"].mean()
        features[f"cmj_peak_power_max_{days}d"] = w_cmj["Peak Power / BM [W/kg]"].max()
    
        features[f"cmj_concentric_force_mean_{days}d"] = w_cmj["Concentric Mean Force [N]"].mean()
        features[f"cmj_concentric_force_max_{days}d"] = w_cmj["Concentric Mean Force [N]"].max()
    
        features[f"cmj_eccentric_force_mean_{days}d"] = w_cmj["Eccentric Mean Force [N]"].mean()
        features[f"cmj_eccentric_force_max_{days}d"] = w_cmj["Eccentric Mean Force [N]"].max()

        return features


In [17]:

r = races_in_range.iloc[0].copy()
features = race_row(r)
f2 = dict()
for k in features.keys():
    f2[k] = [features[k]]
training1 = pd.DataFrame(f2)


for i in range(1, len(races_in_range)):
    r = races_in_range.iloc[i].copy()
    features = race_row(r)
    f2 = dict()
    for k in features.keys():
        f2[k] = [features[k]]
    nextrow = pd.DataFrame(f2)
    training1 = pd.concat([training1, nextrow], ignore_index=True)
training1

,distance_10d,sessions_10d,intensity_anaerobic_capacity_10d,intensity_aerobic_power_10d,intensity_anaerobic_power_10d,intensity_speed_10d,distance_Backstroke_10d,taper_ratio,distance_40d,sessions_40d,...,cmj_eccentric_force_mean_10d,cmj_eccentric_force_max_10d,distance_Freestyle_10d,distance_Freestyle_40d,distance_Butterfly_10d,distance_Butterfly_40d,distance_Medley_10d,distance_Medley_40d,distance_Breaststroke_10d,distance_Breaststroke_40d
0,0,0,0.0,0.0,0.0,0.0,0.0,0.00,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,54450,12,950.0,600.0,200.0,70.0,NaN,25.84,210740,48,...,800.539667,801.810,45300.0,178290.0,NaN,NaN,NaN,NaN,NaN,NaN
2,31450,12,750.0,300.0,700.0,375.0,NaN,18.24,172430,50,...,NaN,NaN,27300.0,148335.0,NaN,NaN,NaN,NaN,NaN,NaN
3,0,0,0.0,0.0,0.0,0.0,NaN,0.00,0,0,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,26645,8,500.0,0.0,400.0,270.0,NaN,16.87,157920,38,...,NaN,NaN,21225.0,131860.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
669,0,0,0.0,0.0,0.0,0.0,NaN,0.00,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN
670,56500,14,225.0,500.0,500.0,765.0,NaN,24.16,233810,54,...,786.913000,788.757,44000.0,195735.0,NaN,NaN,NaN,NaN,NaN,NaN
671,42510,13,600.0,0.0,280.0,350.0,NaN,21.71,195820,51,...,683.778500,688.372,NaN,NaN,1250.0,5275.0,NaN,NaN,NaN,NaN
672,0,0,0.0,0.0,0.0,0.0,NaN,0.00,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN


In [26]:
#Within-Event Modelling to remove race characteristics as variables

#Create a subset of race rows that only include LC 200 freestyle

subset1 = training1[
    (training1["race"] == "200 m Freestyle") &
    (training1["pool_length"] == 50)]

subset1 = subset1.dropna(axis=1, how="all")

subset1.shape

(66, 30)

In [27]:
target = "race_time"

In [28]:
#Feature and Target separation
#x is predictor variables
#y is target variable (race_time)^

x = subset1.drop(columns=[target])
y = subset1[target]

In [29]:
#Simple split (train-test)

from sklearn.model_selection import train_test_split

#test_size 0.2 to allow for 80% training data and 20% testing data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [30]:
#Categorical variable handling / missing values

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

numeric_cols = x.select_dtypes(include = ["number"]).columns
categorical_cols = x.select_dtypes(exclude=["number"]).columns

In [31]:
#Linear Regression Preprocessing

#Missing numerical values replaced with median
numeric_transformer_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))])

#missing categorical values replaced with most frequent
#categorical variable are one-hot coded
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocessor_lr = ColumnTransformer([
    ("num", numeric_transformer_lr, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)])

In [32]:
#MLP Preprocessing (+scaling)

numeric_transformer_mlp = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())])

preprocessor_mlp = ColumnTransformer([
    ("num", numeric_transformer_mlp, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)])

In [33]:
#Linear regression fitting

lr_model = Pipeline([
    ("preprocessor", preprocessor_lr),
    ("model", LinearRegression())])

#Fit with the training dataset

lr_model.fit(x_train, y_train)

#Prediction generated on test dataset

lr_pred = lr_model.predict(x_test)

In [34]:
#MLPRegressor fitting

#Two hidden layers with 50 neurons each, maximum iterations of 1000
mlp_model = Pipeline([
    ("preprocessor", preprocessor_mlp),
    ("model", MLPRegressor(
        hidden_layer_sizes=(50,50),
        max_iter=2000,
        random_state=42))])

#Fit with the training dataset

mlp_model.fit(x_train, y_train)

#Prediction generated on test dataset

mlp_pred = mlp_model.predict(x_test)

In [35]:
#Compare Performance

#Include MAE, RMSE, R2 as quality metrics
def evaluate(y_true, y_pred, name):
    print(f"\n{name}")
    print("MAE:", mean_absolute_error(y_true, y_pred))
    print("RMSE:", mean_squared_error(y_true, y_pred) ** 0.5)
    print("R2:", r2_score(y_true, y_pred))

evaluate(y_test, lr_pred, "Linear Regression")
evaluate(y_test, mlp_pred, "MLP Regressor")


Linear Regression
MAE: 9.169730910238206
RMSE: 12.811796208140263
R2: -2.7099949001142694

MLP Regressor
MAE: 8.265337333659762
RMSE: 10.47607799825096
R2: -1.4805655920378777


In [36]:
print(list(subset1.columns))

['distance_10d', 'sessions_10d', 'intensity_anaerobic_capacity_10d', 'intensity_aerobic_power_10d', 'intensity_anaerobic_power_10d', 'intensity_speed_10d', 'taper_ratio', 'distance_40d', 'sessions_40d', 'intensity_anaerobic_capacity_40d', 'intensity_aerobic_power_40d', 'intensity_anaerobic_power_40d', 'intensity_speed_40d', 'race_phase', 'race_stroke', 'race_distance', 'pool_length', 'race_time', 'race', 'cmj_tests_10d', 'cmj_jump_height_mean_10d', 'cmj_jump_height_max_10d', 'cmj_peak_power_mean_10d', 'cmj_peak_power_max_10d', 'cmj_concentric_force_mean_10d', 'cmj_concentric_force_max_10d', 'cmj_eccentric_force_mean_10d', 'cmj_eccentric_force_max_10d', 'distance_Freestyle_10d', 'distance_Freestyle_40d']
